<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Random SFI Record Number

There are four records numbered 1 through 4. `random.randint(1, record_count)` includes both ends, so every record can be picked. To show that, the runner picks 100 times and counts how often each record came up. Repeats are normal because each pick does not depend on the one before it.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Random SFI Record Number

import random

record_count = 4

record_id = random.randint(1, record_count)
print("SFI record selected for QA:", record_id)

# Evidence that every valid record ID can occur: pick 100 times and count.
counts = {record: 0 for record in range(1, record_count + 1)}
for run in range(100):
    counts[random.randint(1, record_count)] += 1

print()
print("Picks per record over 100 runs:")
for record, count in counts.items():
    print("  Record", record, "->", count)
print("Every record was picked at least once:", all(count > 0 for count in counts.values()))
print("Possible outputs:", list(range(1, record_count + 1)))

## Popcorn Hack 2: Random Structured SFI Car Part

`random.choice(parts)` picks one complete record, so its product name, category, and spec number stay together.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Random SFI Part Record

import random

parts = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

selected_part = random.choice(parts)

print("SFI part selected for testing:")
print("  Product name:", selected_part["product_name"])
print("  Category:    ", selected_part["category"])
print("  Spec number: ", selected_part["spec_number"])

## Popcorn Hack 3: Random SFI API Test

Each test picks a random part **and** a random action, then routes the action to its own backend-style response. With 3 parts and 3 actions there are 9 possible pairs.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Random SFI API Test

import random

parts = [
    {"product_name": "Replacement Flywheels", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "spec_number": "2.1"}
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update"
]

for test in range(1, 6):
    part = random.choice(parts)
    action = random.choice(actions)

    if action == "GET/search":
        response = "200 OK - found spec " + part["spec_number"]
    elif action == "POST/create":
        response = "201 Created - validated " + part["product_name"] + " before saving"
    else:  # PUT/update
        response = "200 OK - updated spec " + part["spec_number"]

    print("Test", test, "|", action, "|", part["product_name"], "->", response)

## Popcorn Hack 4 / Homework: SFI Backend QA Simulator

A reusable simulator that runs `test_count` random tests. Each test picks a random part and action and applies it to the stored spec numbers:

| Action | Spec already stored | Spec not stored |
| --- | --- | --- |
| `GET/search` | 200: record found | 404: not found |
| `POST/create` | 409: duplicate spec, rejected | 201: created and stored |
| `PUT/update` | 200: updated | 404: nothing to update |
| `DELETE/remove` | 200: removed | 404: nothing to remove |

The stored list changes as tests run, so a spec created early in a run is rejected as a duplicate if it is created again later.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - SFI Backend QA Simulator

import random

parts = [
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update",
    "DELETE/remove"
]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5


def run_qa_test(part, action):
    """Apply one action to the backend and return the result."""
    spec = part["spec_number"]
    stored = spec in existing_spec_numbers

    if action == "GET/search":
        if stored:
            return "200 OK: found " + part["product_name"]
        return "404 Not Found: spec " + spec + " is not stored"

    if action == "POST/create":
        if stored:
            return "409 Conflict: spec " + spec + " already exists"
        existing_spec_numbers.append(spec)
        return "201 Created: stored spec " + spec

    if action == "PUT/update":
        if stored:
            return "200 OK: updated spec " + spec
        return "404 Not Found: cannot update spec " + spec

    if action == "DELETE/remove":
        if stored:
            existing_spec_numbers.remove(spec)
            return "200 OK: removed spec " + spec
        return "404 Not Found: cannot remove spec " + spec

    return "400 Bad Request: unknown action " + action


def run_simulator(count):
    print("Starting specs:", existing_spec_numbers)
    print()
    for test_number in range(1, count + 1):
        part = random.choice(parts)
        action = random.choice(actions)
        result = run_qa_test(part, action)
        print("Test", test_number, "|", action, "|", part["spec_number"], part["product_name"])
        print("   ->", result)
    print()
    print("Ending specs:", existing_spec_numbers)
    print("No duplicate specs stored:", len(existing_spec_numbers) == len(set(existing_spec_numbers)))


run_simulator(test_count)